# Build A Simple Retrieval-Augmented QA Agent

In this tutorial we will do the folloing:
- **Prepare a Knowledge Base**: We’ll use a small list of text documents as our "knowledge base"
- **Build a Vector Index**: We will use a TF-IDF vectorizer (from scikit-learn) to index these documents
- **Retrieve Relevant Document**: Given a user question, we’ll vectorize the query and find which document is most similar (contains the most relevant words).
- **Generate an Answer**




## Step 1: Knowledge Base

In [ ]:
# a small set of text documents
docs = [
  "Albert Einstein was born in 1879 in Ulm, Germany. He developed the theory of relativity and won the Nobel Prize in Physics in 1921.",
  "The Eiffel Tower is located in Paris, France. It was completed in 1889 and is one of the most recognized structures in the world.",
  "Python is a high-level programming language created by Guido van Rossum and first released in 1991. It emphasizes code readability and has a wide range of applications."
]

In [ ]:
print(f"I have {len(docs)} documents")

## Step 2: Indexing

We'll use **TfidfVectorizer** to create a simple vector representation for each document. Then we transform each document into a vector and store the results.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer and fit it on our documents
# Common words like "the", "is", "and" are removed
vectorizer = TfidfVectorizer(stop_words = 'english')

# Vectorizer learns the vocabulary of the documents
# fit():
# Identifies all unique terms (words) across the input documents, excluding stop words
# Compute the Inverse Document Frequency (IDF) for each word
# IDF(t) = log(N / 1 + n(t))
# N = total number of documents
# n(t) = number of documents that contain term t

# transform():
# Compute the term frequency (TF) for each term in each document
# TF(t, d) = n(t, d) / sum(n(t, d))
# n(t, d) = number of times term t appears in document d
# sum(n(t, d)) = total number of terms in document d

# Multiples TF and IDF together to get the TF-IDF score
# Output is a sparse matrix of shape (num_docs, num_terms)
# Row = Document
# Column = Unique word in vocab
# Higher the score, more important the word is for that document
doc_vectors = vectorizer.fit_transform(docs)

## Step 3: Retrieval Function

Given a user query:
- Transform the query into the TF-IDF vector space. 
- Compute similarity (for TF-IDF, a dot product of normalized vectors is a cosine similarity). 
- Find the document with the highest similarity score.
- Return that document as the relevant context.

In [ ]:
import numpy as np

def retrieve_top_doc(query):
  # Vectorize the query using the same vectorizer
  # Ensure the query is represented in the same feature space as the documents
  # If using a different vectorizer, the feature indices wouldn't align
  # making similarity comparision invalid
  query_vec = vectorizer.transform([query])

  # Compute similarity scores with each document vector
  # doc_vectors = (num_docs, num_terms)
  # query_vec.T = (num_terms, 1)
  # scores = (num_docs, 1)
  # This dot product represents cosine similarity between query and each document
  scores = (doc_vectors * query_vec.T).toarray()

  # Finds the index of highest similarity score, most relevant document
  top_index = np.argmax(scores)

  # Return both document and its index for reference
  return docs[top_index], top_index

In [ ]:
# Example queries
questions = [
  "When was Albert Einstein born?",
  "Where is the Eiffel Tower located?",
  "Who created Python and when was it first released?"
]

# Retrieve and display the top document for each query
contexts = []

for q in questions:
  top_doc, idx = retrieve_top_doc(q)
  contexts.append(top_doc)
  print(f"Query: {q}")
  print(f"Retrieved Document {idx}: {top_doc[:60]}...") # printing first 60 chars for brevity
  print()

## Step 4: Generating the Answer

In [ ]:
def build_prompt(question, retrieved_text):
  system = "You are a helpful RAG assistant. Use ONLY the provided context. If unsure, say you don't know."
  user = f"""Answer the question using the context.

  Context:
  {retrieved_text}

  Question: {question}
  """
  # Simple chat-like format many instruct models accept:
  return f"[SYSTEM]\n{system}\n[/SYSTEM]\n[USER]\n{user}\n[/USER]\n[ASSISTANT]\n"

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_API_TOKEN")
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
  "Authorization": f"Bearer {HF_TOKEN}",
}

# Function to query HF Inference API
def query_hf(prompt):
  payload = {
    "messages": [
      {
        "role": "user",
        "content": prompt
      }
    ],
    "model": "meta-llama/Llama-3.1-8B-Instruct:novita"
  }
  
  response = requests.post(API_URL, headers = headers, json = payload)
  result = response.json()

  # Return the first generated text from the response
  return result["choices"][0]["message"]["content"]

In [ ]:
prompt = build_prompt(questions[1], contexts[0])
answer = query_hf(prompt)
print(prompt)
print(answer)